# Vendedores da Olist: consultas em SQL

Neste notebook uso SQL (SQLite) para analisar os vendedores do marketplace da Olist. As perguntas são:

1. Quantos vendedores existem e onde estão?
2. Quem mais vendeu?
3. Quanto do faturamento está concentrado em poucos vendedores?
4. Os vendedores que mais atrasam entregas também têm notas piores?

Os dados são os CSVs da Olist (Kaggle), carregados no banco pelo script `criar_banco.py`.

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../olist.db")

def consulta(sql):
    return pd.read_sql(sql, con)

In [2]:
consulta("""
SELECT COUNT(*) AS vendedores_cadastrados,
       (SELECT COUNT(DISTINCT seller_id) FROM order_items) AS vendedores_com_venda
FROM sellers
""")

,vendedores_cadastrados,vendedores_com_venda
0,3095,3095


In [3]:
consulta("""
SELECT seller_state AS estado, COUNT(*) AS vendedores
FROM sellers
GROUP BY seller_state
ORDER BY vendedores DESC
""")

,estado,vendedores
0,SP,1849
1,PR,349
2,MG,244
3,SC,190
4,RJ,171
5,RS,129
6,GO,40
7,DF,30
8,ES,23
9,BA,19


Todos os 3.095 vendedores cadastrados aparecem em pelo menos um item vendido, então não precisei filtrar vendedores sem venda nas próximas consultas.

Por estado, os vendedores se concentram bastante: SP tem 1.849 (cerca de 60% do total) e os seis primeiros estados (SP, PR, MG, SC, RJ e RS) juntos reúnem cerca de 95%. É mais concentrado do que os clientes, onde SP tinha 41,9%. Os estados do Norte e do Nordeste têm poucos vendedores, quase sempre menos de dez cada.

In [5]:
consulta("""
SELECT oi.seller_id,
       s.seller_state AS estado,
       COUNT(DISTINCT oi.order_id) AS pedidos,
       ROUND(SUM(oi.price), 2) AS valor_vendido
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN sellers s ON s.seller_id = oi.seller_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY oi.seller_id, s.seller_state
ORDER BY valor_vendido DESC
LIMIT 10
""")

,seller_id,estado,pedidos,valor_vendido
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1131,229237.63
1,53243585a1d6dc2643021fd1853d8905,BA,358,222776.05
2,4a3ca9315b744ce9f8e9374361493884,SP,1804,200326.12
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,584,192842.13
4,7c67e1448b00f6e969d365cea6b010ab,SP,982,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,SP,332,172583.88
6,da8622b14eb17ae2831f4ac5b9dab84a,SP,1314,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,SP,1159,141715.54
8,1025f0e2d44d7041d6cf58b6550e0bfa,SP,915,138968.55
9,955fee9216a65b617aa5c0531780ce60,SP,1286,135159.70


O ranking por valor vendido é dominado por vendedores de SP: nove dos dez primeiros são de lá. A exceção é o segundo colocado, da BA, com R$ 222 mil, quase igual ao primeiro (R$ 229 mil).

Olhando o valor por pedido, os perfis são bem diferentes. O primeiro vendeu em torno de R$ 203 por pedido, o da BA chegou a quase R$ 620 com apenas 358 pedidos, e o terceiro tem o maior volume do top 10 (1.804 pedidos) com cerca de R$ 111 por pedido. Ou seja, dá para chegar ao topo vendendo muito de coisas baratas ou pouco de coisas caras, e o ranking somente por valor esconde essa diferença.

In [6]:
consulta("""
WITH vendas AS (
    SELECT oi.seller_id, SUM(oi.price) AS valor
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY oi.seller_id
),
acumulado AS (
    SELECT seller_id,
           valor,
           ROW_NUMBER() OVER (ORDER BY valor DESC) AS posicao,
           SUM(valor) OVER (
               ORDER BY valor DESC
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) AS valor_acumulado
    FROM vendas
)
SELECT posicao,
       ROUND(valor, 2) AS valor,
       ROUND(100.0 * valor_acumulado / (SELECT SUM(valor) FROM vendas), 1) AS pct_acumulado
FROM acumulado
WHERE posicao IN (1, 10, 50, 100, 300, 500, 1000, 2000)
ORDER BY posicao
""")

,posicao,valor,pct_acumulado
0,1,229237.63,1.7
1,10,135159.70,13.2
2,50,43517.50,33.0
3,100,25080.03,45.3
4,300,9669.80,67.1
5,500,5966.54,78.4
6,1000,2061.37,91.6
7,2000,371.20,98.8


O faturamento é concentrado, mas sem um vendedor dominante: o maior sozinho responde por 1,7% do total e os 10 maiores por 13,2%. A concentração aparece em grupos maiores. Os 100 maiores vendedores (cerca de 3% dos 3.095) reúnem 45,3% do valor vendido, e os 500 maiores (cerca de 16%) chegam a 78,4%.

Na outra ponta, os vendedores depois da posição 1.000 somam só 8,4% do total, e os que estão depois da posição 2.000 dividem 1,2%. Ou seja, boa parte dos vendedores da base vende muito pouco.

In [7]:
vend = consulta("""
WITH base AS (
    SELECT DISTINCT oi.seller_id, oi.order_id
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
),
avaliacao AS (
    SELECT order_id,
           review_score,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY review_answer_timestamp DESC
           ) AS rn
    FROM order_reviews
)
SELECT b.seller_id,
       s.seller_state AS estado,
       COUNT(*) AS pedidos,
       ROUND(100.0 * SUM(o.order_delivered_customer_date > o.order_estimated_delivery_date) / COUNT(*), 1) AS pct_atrasados,
       ROUND(AVG(a.review_score), 2) AS nota_media
FROM base b
JOIN orders o ON o.order_id = b.order_id
JOIN sellers s ON s.seller_id = b.seller_id
LEFT JOIN avaliacao a ON a.order_id = b.order_id AND a.rn = 1
GROUP BY b.seller_id, s.seller_state
HAVING COUNT(*) >= 50
""")
vend.shape[0]

425

In [8]:
vend.sort_values("pct_atrasados", ascending=False).head(10)

,seller_id,estado,pedidos,pct_atrasados,nota_media
143,54965bbe3e4f07ae045b90b0b8541f52,PR,73,30.1,3.14
285,a49928bcdf77c55c6d6e05e09a9b4ca5,SP,96,26.0,3.03
316,beadbee30901a7f61d031b6b686095ad,SP,64,25.0,3.91
20,06a2c3af7b3aee5d69171b0e14f0ee87,MA,389,23.1,4.01
331,cac4c8e7b1ca6252d8f20b2fc1a2e4af,SP,74,23.0,3.56
62,1ca7077d890b907f89be8c954a02686a,SP,108,22.2,2.39
159,6039e27294dc75811c0d8a39069f52c0,SP,63,22.2,3.73
191,712e6ed8aa4aa1fa65dab41fed5737e4,SC,77,22.1,3.39
389,ea566164622c6b439516ab18062c42cd,SP,50,22.0,3.78
347,d13e50eaa47b4cbe9eb81465865d8cfc,SP,66,21.2,4.82


In [9]:
vend.sort_values("nota_media").head(10)

,seller_id,estado,pedidos,pct_atrasados,nota_media
62,1ca7077d890b907f89be8c954a02686a,SP,108,22.2,2.39
90,2eb70248d66e0e3ef83659f71b244378,SP,187,13.9,2.79
269,972d0f9cf61b499a4812cf0bfa3ad3c4,SC,81,11.1,3.03
285,a49928bcdf77c55c6d6e05e09a9b4ca5,SP,96,26.0,3.03
143,54965bbe3e4f07ae045b90b0b8541f52,PR,73,30.1,3.14
313,bbad7e518d7af88a0897397ffdca1979,SP,68,16.2,3.18
132,5058e8c1e82653974541e83690655b4a,SP,62,12.9,3.25
230,8444e55c1f13cd5c179851e5ca5ebd00,MG,92,6.5,3.37
182,6fd52c528dcb38be2eea044946b811f8,SP,67,13.4,3.39
191,712e6ed8aa4aa1fa65dab41fed5737e4,SC,77,22.1,3.39


In [10]:
vend[["pct_atrasados", "nota_media"]].corr()

,pct_atrasados,nota_media
pct_atrasados,1.000000,-0.501228
nota_media,-0.501228,1.000000


Para comparar atrasos e notas por vendedor, usei apenas os 425 que tiveram pelo menos 50 pedidos entregues, porque com poucos pedidos o percentual de atraso e a nota média variam demais. Mantive as regras da [análise em Python](https://github.com/paulorcmj/analise-vendas-olist) que fiz com os mesmos dados: somente pedidos entregues, atraso quando a entrega passa da data estimada e, em pedidos com mais de uma avaliação, a mais recente.

Os vendedores que mais atrasam possuem entre 21% e 30% dos pedidos fora do prazo, e quatro deles também estão entre os 10 com pior nota. A correlação entre percentual de atrasos e nota média é de -0,50: quem atrasa mais tende a ter nota menor, mas a relação é moderada. Há vendedores com mais de 20% de atraso e nota acima de 4, e vendedores com pouco atraso e nota baixa, o que sugere que a nota depende de outros fatores além do prazo, como, possivelmente, o produto em si.

In [11]:
consulta("""
WITH por_vendedor AS (
    SELECT oi.seller_id,
           COUNT(DISTINCT oi.order_id) AS pedidos,
           SUM(oi.price) AS valor
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY oi.seller_id
)
SELECT SUM(pedidos >= 50) AS vendedores,
       ROUND(100.0 * SUM(CASE WHEN pedidos >= 50 THEN valor ELSE 0 END) / SUM(valor), 1) AS pct_do_valor
FROM por_vendedor
""")

,vendedores,pct_do_valor
0,425,68.1


Os 425 vendedores com pelo menos 50 pedidos entregues respondem por 68,1% do valor vendido nos pedidos entregues. O filtro deixa de fora a maioria dos vendedores, mas também cerca de um terço do valor, portanto, as conclusões sobre atraso e nota valem para os vendedores de maior volume.

In [12]:
consulta("""
SELECT s.seller_state AS estado,
       COUNT(DISTINCT oi.seller_id) AS vendedores,
       ROUND(SUM(oi.price), 2) AS valor_vendido,
       ROUND(100.0 * SUM(oi.price) / SUM(SUM(oi.price)) OVER (), 1) AS pct_do_valor
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN sellers s ON s.seller_id = oi.seller_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY s.seller_state
ORDER BY valor_vendido DESC
LIMIT 10
""")

,estado,vendedores,valor_vendido,pct_do_valor
0,SP,1822,8687740.35,64.4
1,PR,344,1256874.01,9.3
2,MG,240,1001944.76,7.4
3,RJ,170,837466.37,6.2
4,SC,186,624002.96,4.6
5,RS,129,377726.85,2.8
6,BA,19,285332.57,2.1
7,DF,30,97100.48,0.7
8,PE,9,91463.95,0.7
9,GO,40,66251.72,0.5


São Paulo concentra 64,4% do valor vendido, e os seis primeiros estados (SP, PR, MG, RJ, SC e RS) chegam a 94,7%. Aqui só contam vendedores com ao menos um pedido que não foi cancelado nem ficou indisponível, por isso SP aparece com 1.822 vendedores e não com 1.849.

A Bahia tem apenas 19 vendedores e 2,1% do valor, mas cerca de 78% do que o estado vendeu vem de um único vendedor, o segundo colocado do ranking geral. Sem ele, a participação do estado seria bem menor.

## Conclusão

Os vendedores da Olist estão concentrados no Sudeste e no Sul: SP tem 59,7% deles e 64,4% do valor vendido. No topo do ranking aparecem perfis diferentes, com vendedores de muito volume e ticket baixo e outros de pouco volume e ticket alto, e o faturamento se concentra em grupos de algumas centenas de vendedores: os 100 maiores respondem por 45,3% do total.

Entre os vendedores com pelo menos 50 pedidos, quem atrasa mais tende a ter nota menor (correlação de -0,50), mas a relação é moderada e há vendedores que fogem dela.

Algumas limitações: o valor considera apenas o preço dos itens, sem frete, e exclui pedidos cancelados e indisponíveis; o filtro de 50 pedidos deixa de fora cerca de um terço do valor; e a ideia de que o produto influencia a nota não foi testada aqui.